In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/function_vectors_eval'
for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories and common non-essential directories
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__', 'node_modules', '.git']]
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

function_vectors_eval/
  .gitignore
  fv_overview.png
  documentation.pdf
  plan.md
  CodeWalkthrough.md
  fv_environment.yml
  src/
    portability_eval.py
    test_numheads.py
    compute_indirect_effect.py
    vocab_reconstruction.py
    __init__.py
    compute_avg_hidden_state.py
    natural_text_eval.py
    evaluate_function_vector.py
    compute_average_activations.py
    utils/
      eval_utils.py
      prompt_utils.py
      intervention_utils.py
      extract_utils.py
      __init__.py
      model_utils.py
    eval_scripts/
      eval_fv.sh
      eval_numheads.sh
      eval_template_portability.sh
      eval_avg_hs.sh
      template.sh
      fv_eval_sweep.py
  notebooks/
    fv_demo.ipynb
  dataset_files/
    README.md
    extractive/
      color_v_animal_5.json
      adjective_v_verb_5.json
      alphabetically_last_5.json
      choose_middle_of_5.json
      animal_v_object_3.json
      conll2003_organization.json
      conll2003_person.json
      fruit_v_animal_3.json
      a

# Function Vectors in Large Language Models - Replication

This notebook replicates the core experiment from the Function Vectors paper:
- Compute task-conditioned mean activations from ICL prompts
- Extract function vectors by summing outputs of top causal attention heads
- Test function vectors in zero-shot and shuffled-label contexts

## Goal
Verify that function vectors can trigger task execution even without proper ICL demonstrations.

In [3]:
# Create the evaluation/replications directory
import os
repo_path = '/net/scratch2/smallyan/function_vectors_eval'
replication_dir = os.path.join(repo_path, 'evaluation', 'replications')
os.makedirs(replication_dir, exist_ok=True)
print(f"Created directory: {replication_dir}")

Created directory: /net/scratch2/smallyan/function_vectors_eval/evaluation/replications


In [4]:
# Check CUDA availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA H100 PCIe
Memory: 85.0 GB


## Setup and Imports

Import required libraries and set up environment.

In [5]:
# Core imports
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Disable gradient computation for inference
torch.set_grad_enabled(False)

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print("Imports complete and seeds set")

Imports complete and seeds set


## Data Loading Utilities

Reimplementing dataset loading and prompt creation from the plan understanding.

In [6]:
# ICL Dataset class - reimplemented from understanding of the codebase
class ICLDataset:
    """Dataset class for in-context learning containing input-output pairs."""
    
    def __init__(self, data):
        if isinstance(data, str):
            self.data = pd.read_json(data)
        elif isinstance(data, dict):
            self.data = pd.DataFrame(data)
        self.data = self.data[['input', 'output']]
    
    def __getitem__(self, idx):
        if isinstance(idx, int):
            return self.data.iloc[idx].to_dict()
        elif isinstance(idx, (slice, list, np.ndarray)):
            return self.data.iloc[idx].to_dict(orient='list')
        elif isinstance(idx, str):
            return self.data[idx].to_list()
        raise ValueError(f"Invalid index type: {type(idx)}")
    
    def __len__(self):
        return len(self.data)


def split_dataset(dataset, test_size=0.3, seed=32):
    """Split dataset into train/valid/test splits."""
    train, valid = train_test_split(dataset.data, test_size=test_size, random_state=seed)
    test, valid = train_test_split(valid, test_size=test_size, random_state=seed)
    
    return {
        'train': ICLDataset(train.to_dict(orient='list')),
        'valid': ICLDataset(valid.to_dict(orient='list')),
        'test': ICLDataset(test.to_dict(orient='list'))
    }


def load_task_dataset(task_name, root_dir='/net/scratch2/smallyan/function_vectors_eval/dataset_files', 
                      test_size=0.3, seed=32):
    """Load a task dataset by name."""
    # Check abstractive and extractive folders
    for folder in ['abstractive', 'extractive']:
        path = os.path.join(root_dir, folder, f'{task_name}.json')
        if os.path.exists(path):
            dataset = ICLDataset(path)
            return split_dataset(dataset, test_size=test_size, seed=seed)
    
    raise FileNotFoundError(f"Dataset '{task_name}' not found")


# Test loading
dataset = load_task_dataset('antonym')
print(f"Train size: {len(dataset['train'])}")
print(f"Valid size: {len(dataset['valid'])}")
print(f"Test size: {len(dataset['test'])}")
print(f"Sample pair: {dataset['train'][0]}")

Train size: 1678
Valid size: 216
Test size: 504
Sample pair: {'input': 'hardware', 'output': 'software'}


In [7]:
# Prompt creation utilities - reimplemented from understanding

def create_prompt_data(word_pairs, query_pair=None, shuffle_labels=False, 
                       prepend_bos=False, prepend_space=True,
                       prefixes=None, separators=None):
    """
    Create structured prompt data from word pairs.
    
    Args:
        word_pairs: dict with 'input' and 'output' lists
        query_pair: dict with 'input' and 'output' for the test query
        shuffle_labels: whether to shuffle the output labels
        prepend_bos: whether to add BOS token
        prepend_space: whether to prepend space to tokens
        prefixes: dict of prefix strings for input/output/instructions
        separators: dict of separator strings
    """
    if prefixes is None:
        prefixes = {'input': 'Q:', 'output': 'A:', 'instructions': ''}
    if separators is None:
        separators = {'input': '\n', 'output': '\n\n', 'instructions': ''}
    
    # Handle BOS token
    if prepend_bos:
        prefixes = {k: ('<|endoftext|>' + v if k == 'instructions' else v) 
                   for k, v in prefixes.items()}
    
    prompt_data = {
        'prefixes': prefixes,
        'separators': separators,
        'instructions': '',
        'query_target': None,
        'examples': []
    }
    
    # Process query pair
    if query_pair is not None:
        q_in = query_pair['input']
        q_out = query_pair['output']
        if isinstance(q_in, list):
            q_in = q_in[0]
        if isinstance(q_out, list):
            q_out = q_out[0]
        if prepend_space:
            q_in = ' ' + str(q_in)
            q_out = ' ' + str(q_out)
        prompt_data['query_target'] = {'input': q_in, 'output': q_out}
    
    # Process examples
    inputs = word_pairs.get('input', [])
    outputs = word_pairs.get('output', [])
    
    if shuffle_labels and len(outputs) > 0:
        outputs = np.random.permutation(outputs).tolist()
    
    for inp, out in zip(inputs, outputs):
        if prepend_space:
            inp = ' ' + str(inp)
            out = ' ' + str(out)
        prompt_data['examples'].append({'input': inp, 'output': out})
    
    return prompt_data


def build_prompt(prompt_data, query=None):
    """Build the actual prompt string from prompt data."""
    prompt = ''
    
    # Add instructions
    prompt += prompt_data['prefixes']['instructions']
    prompt += prompt_data['instructions']
    prompt += prompt_data['separators']['instructions']
    
    # Add examples
    for example in prompt_data['examples']:
        prompt += prompt_data['prefixes']['input']
        prompt += example['input']
        prompt += prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output']
        prompt += example['output']
        prompt += prompt_data['separators']['output']
    
    # Add query
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    
    if query is not None:
        if isinstance(query, list):
            query = query[0]
        prompt += prompt_data['prefixes']['input']
        prompt += query
        prompt += prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output']
    
    return prompt


# Test prompt creation
test_pairs = dataset['train'][:5]
test_query = dataset['test'][0]
prompt_data = create_prompt_data(test_pairs, query_pair=test_query, prepend_bos=True)
prompt = build_prompt(prompt_data)
print("Sample prompt:")
print(repr(prompt[:300]))

Sample prompt:
'<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: swift\nA:'


## Model Loading

Load GPT-J model and set up model configuration for activation extraction.

In [8]:
# Model loading utility - reimplemented
def load_model_and_tokenizer(model_name, device='cuda'):
    """Load model, tokenizer, and create model config."""
    print(f"Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        low_cpu_mem_usage=True
    ).to(device)
    
    # Create model config based on model type
    if 'gpt-j' in model_name.lower():
        config = {
            'n_heads': model.config.n_head,
            'n_layers': model.config.n_layer,
            'resid_dim': model.config.n_embd,
            'name_or_path': model.config.name_or_path,
            'attn_hook_names': [f'transformer.h.{l}.attn.out_proj' for l in range(model.config.n_layer)],
            'layer_hook_names': [f'transformer.h.{l}' for l in range(model.config.n_layer)],
            'prepend_bos': False
        }
    else:
        raise NotImplementedError(f"Model {model_name} not yet supported")
    
    return model, tokenizer, config

# Load GPT-J
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_model_and_tokenizer(model_name)
print(f"Model loaded: {model_config['n_layers']} layers, {model_config['n_heads']} heads")
print(f"Residual dimension: {model_config['resid_dim']}")

Loading model: EleutherAI/gpt-j-6b


Exception ignored in: <function tqdm.__del__ at 0x7fc1003a9260>
Traceback (most recent call last):
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

Model loaded: 28 layers, 16 heads
Residual dimension: 4096


## Token Label Utilities

Implement utilities for labeling tokens in ICL prompts for activation extraction.

In [9]:
# Token labeling utilities for activation extraction

def get_prompt_parts_with_labels(prompt_data, query=None):
    """Generate token labels for each part of an ICL prompt."""
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    if isinstance(query, list):
        query = query[0]
    
    n_examples = len(prompt_data['examples'])
    
    # Build prompt parts and their labels
    parts = [prompt_data['prefixes']['instructions'], 
             prompt_data['instructions'], 
             prompt_data['separators']['instructions']]
    labels = ['bos_token', 'instructions_token', 'separator_token']
    
    for i, example in enumerate(prompt_data['examples']):
        parts.extend([
            prompt_data['prefixes']['input'], example['input'], prompt_data['separators']['input'],
            prompt_data['prefixes']['output'], example['output'], prompt_data['separators']['output']
        ])
        labels.extend([
            'structural_token', f'demonstration_{i+1}_token', 'separator_token',
            'structural_token', f'demonstration_{i+1}_label_token', 'separator_token'
        ])
    
    # Query part
    parts.extend([
        prompt_data['prefixes']['input'], query, prompt_data['separators']['input'],
        prompt_data['prefixes']['output']
    ])
    labels.extend([
        'query_structural_token', 'query_demonstration_token', 
        'query_separator_token', 'query_structural_token'
    ])
    
    return parts, labels


def extend_labels_to_tokens(parts, labels, tokenizer, prepend_bos=False):
    """Extend labels across tokenized text."""
    token_labels = ['bos_token'] if prepend_bos else []
    prompt_builder = ''
    
    for part, label in zip(parts, labels):
        if len(part) == 0:
            continue
        pre_len = len(tokenizer.tokenize(prompt_builder))
        prompt_builder += part
        post_len = len(tokenizer.tokenize(prompt_builder))
        n_tokens = post_len - pre_len
        
        if n_tokens == 0 and len(token_labels) > 0:
            token_labels[-1] = label
        else:
            token_labels.extend([label] * n_tokens)
    
    return token_labels


def get_token_labels(prompt_data, tokenizer, query=None, prepend_bos=False):
    """Get token labels for an ICL prompt."""
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    if isinstance(query, list):
        query = query[0]
    
    parts, labels = get_prompt_parts_with_labels(prompt_data, query)
    token_labels = extend_labels_to_tokens(parts, labels, tokenizer, prepend_bos)
    prompt_string = build_prompt(prompt_data, query)
    tokens = [tokenizer.decode(x) for x in tokenizer(prompt_string).input_ids]
    
    labeled = list(zip(range(len(tokens)), tokens, token_labels))
    return labeled, prompt_string


def get_dummy_labels(n_examples, tokenizer, model_config):
    """Get ground-truth labels for a dummy prompt with single-token words."""
    prepend_bos = not model_config['prepend_bos']
    dummy_data = create_prompt_data(
        {'input': ['a'] * n_examples, 'output': ['a'] * n_examples},
        query_pair={'input': 'a', 'output': 'a'},
        prepend_bos=prepend_bos
    )
    labeled, _ = get_token_labels(dummy_data, tokenizer, prepend_bos=model_config['prepend_bos'])
    return [(x[0], x[-1]) for x in labeled]


def compute_index_mapping(token_labels, gt_labels):
    """Map token indices to ground truth positions for averaging."""
    demo_tokens = [(i, t, l) for i, t, l in token_labels if 'demo' in l]
    df = pd.DataFrame(demo_tokens, columns=['idx', 'token', 'label'])
    ranges = df.groupby('label')['idx'].agg(lambda x: (x.min(), x.max()))
    dup_labels = [l for l, (lo, hi) in ranges.items() if hi > lo]
    dup_ranges = ranges[ranges.index.isin(dup_labels)].to_dict()
    
    dup_indices = df[df.duplicated('label')]['idx'].values
    idx_map = {k: v[0] for (k, v) in zip(
        [x[0] for x in token_labels if x[0] not in dup_indices], gt_labels)}
    
    return idx_map, dup_ranges


# Test token labeling
test_prompt_data = create_prompt_data(dataset['train'][:5], query_pair=dataset['test'][0], prepend_bos=True)
labeled, prompt_str = get_token_labels(test_prompt_data, tokenizer, prepend_bos=model_config['prepend_bos'])
print(f"Total tokens: {len(labeled)}")
print("First 10 labeled tokens:")
for item in labeled[:10]:
    print(f"  {item}")

Total tokens: 52
First 10 labeled tokens:
  (0, '<|endoftext|>', 'bos_token')
  (1, 'Q', 'structural_token')
  (2, ':', 'structural_token')
  (3, ' hardware', 'demonstration_1_token')
  (4, '\n', 'separator_token')
  (5, 'A', 'structural_token')
  (6, ':', 'structural_token')
  (7, ' software', 'demonstration_1_label_token')
  (8, '\n', 'separator_token')
  (9, '\n', 'structural_token')


## Activation Extraction

Extract mean attention head activations from ICL prompts - the core of function vector computation.

In [10]:
# Activation extraction - reimplemented

def gather_attention_activations(prompt_data, model, tokenizer, model_config, dummy_labels):
    """Extract attention activations for an ICL prompt."""
    query = prompt_data['query_target']['input']
    token_labels, prompt_string = get_token_labels(prompt_data, tokenizer, query, 
                                                    prepend_bos=model_config['prepend_bos'])
    
    idx_map, idx_avg = compute_index_mapping(token_labels, dummy_labels)
    
    inputs = tokenizer([prompt_string], return_tensors='pt').to(model.device)
    
    # Hook attention output projections
    with TraceDict(model, layers=model_config['attn_hook_names'], 
                   retain_input=True, retain_output=False) as td:
        model(**inputs)
    
    return td, idx_map, idx_avg


def split_by_head(activations, model_config):
    """Split activations by attention head."""
    n_heads = model_config['n_heads']
    head_dim = model_config['resid_dim'] // n_heads
    new_shape = activations.size()[:-1] + (n_heads, head_dim)
    return activations.view(*new_shape)


def compute_mean_activations(dataset, model, model_config, tokenizer, 
                             n_icl_examples=10, n_trials=100, shuffle_labels=False):
    """
    Compute mean attention head activations across many ICL prompts.
    This is the key step for extracting function vectors.
    """
    n_layers = model_config['n_layers']
    n_heads = model_config['n_heads']
    head_dim = model_config['resid_dim'] // n_heads
    
    # Get dummy labels for index mapping
    dummy_labels = get_dummy_labels(n_icl_examples, tokenizer, model_config)
    n_tokens = len(dummy_labels)
    
    # Storage for activations
    storage = torch.zeros(n_trials, n_layers, n_heads, n_tokens, head_dim)
    
    # Whether to add BOS token
    prepend_bos = not model_config['prepend_bos']
    
    valid_indices = np.arange(len(dataset['valid']))
    
    for trial in tqdm(range(n_trials), desc="Computing mean activations"):
        # Sample ICL examples
        train_idx = np.random.choice(len(dataset['train']), n_icl_examples, replace=False)
        word_pairs = dataset['train'][train_idx]
        
        # Sample test query
        test_idx = np.random.choice(valid_indices, 1, replace=False)
        test_pair = dataset['valid'][test_idx]
        
        prompt_data = create_prompt_data(word_pairs, query_pair=test_pair, 
                                         prepend_bos=prepend_bos, shuffle_labels=shuffle_labels)
        
        td, idx_map, idx_avg = gather_attention_activations(
            prompt_data, model, tokenizer, model_config, dummy_labels)
        
        # Stack activations from all layers, split by head
        stacked = torch.vstack([
            split_by_head(td[layer].input, model_config) 
            for layer in model_config['attn_hook_names']
        ]).permute(0, 2, 1, 3)  # (layers, heads, tokens, head_dim)
        
        # Filter to mapped token positions
        filtered = stacked[:, :, list(idx_map.keys())]
        
        # Average multi-token words
        for (i, j) in idx_avg.values():
            filtered[:, :, idx_map[i]] = stacked[:, :, i:j+1].mean(dim=2)
        
        storage[trial] = filtered
    
    mean_activations = storage.mean(dim=0)
    return mean_activations

print("Mean activation computation function defined")

Mean activation computation function defined


## Function Vector Computation

Compute the function vector by summing outputs of top causal attention heads.

According to the plan, the universal top heads for GPT-J were pre-computed via causal mediation analysis.

In [11]:
# Function vector computation using universal top heads

# Pre-computed universal top heads for GPT-J from causal mediation analysis
# These heads have highest average indirect effect across diverse ICL tasks
GPTJ_UNIVERSAL_HEADS = [
    (15, 5, 0.0587), (9, 14, 0.0584), (12, 10, 0.0526), (8, 1, 0.0445), (11, 0, 0.0445),
    (13, 13, 0.019), (8, 0, 0.0184), (14, 9, 0.016), (9, 2, 0.0127), (24, 6, 0.0113),
    (15, 11, 0.0092), (6, 6, 0.0069), (14, 0, 0.0068), (17, 8, 0.0068), (21, 2, 0.0067),
    (10, 11, 0.0066), (11, 2, 0.0057), (17, 0, 0.0054), (20, 11, 0.0051), (23, 0, 0.0047),
    (20, 0, 0.0046), (15, 7, 0.0045), (27, 2, 0.0045), (21, 15, 0.0044), (11, 4, 0.0044),
    (18, 6, 0.0043), (9, 6, 0.0042), (4, 12, 0.004), (11, 15, 0.004), (20, 2, 0.0036),
    (10, 0, 0.0035), (16, 9, 0.0031), (11, 14, 0.0031), (12, 4, 0.003), (9, 7, 0.003),
    (18, 3, 0.003), (19, 5, 0.003), (22, 5, 0.0027), (25, 3, 0.0026), (18, 9, 0.0025)
]


def compute_function_vector(mean_activations, model, model_config, n_top_heads=10):
    """
    Compute function vector by summing outputs of top causal attention heads.
    
    The function vector is computed as:
    FV = sum_{(L,H) in top_heads} O_L * mean_activation[L,H,-1]
    
    where O_L is the output projection matrix for layer L.
    """
    resid_dim = model_config['resid_dim']
    n_heads = model_config['n_heads']
    head_dim = resid_dim // n_heads
    device = model.device
    
    top_heads = GPTJ_UNIVERSAL_HEADS[:n_top_heads]
    
    function_vector = torch.zeros((1, 1, resid_dim)).to(device)
    T = -1  # Use last token position
    
    for layer, head, score in top_heads:
        # Get output projection for this layer
        out_proj = model.transformer.h[layer].attn.out_proj
        
        # Create input with only this head's activation
        x = torch.zeros(resid_dim)
        x[head * head_dim:(head + 1) * head_dim] = mean_activations[layer, head, T]
        
        # Project through output projection
        d_out = out_proj(x.reshape(1, 1, resid_dim).to(device).to(model.dtype))
        function_vector += d_out
    
    function_vector = function_vector.to(model.dtype)
    function_vector = function_vector.reshape(1, resid_dim)
    
    return function_vector, top_heads


print(f"Using top {len(GPTJ_UNIVERSAL_HEADS)} pre-computed universal heads")
print(f"Top 5 heads: {GPTJ_UNIVERSAL_HEADS[:5]}")

Using top 40 pre-computed universal heads
Top 5 heads: [(15, 5, 0.0587), (9, 14, 0.0584), (12, 10, 0.0526), (8, 1, 0.0445), (11, 0, 0.0445)]


## Intervention Utilities

Functions for adding the function vector to model hidden states during inference.

In [12]:
# Intervention utilities for function vector experiments

def create_fv_intervention(edit_layer, fv_vector, device, token_idx=-1):
    """Create intervention function that adds FV to hidden states."""
    def intervention_fn(output, layer_name):
        current_layer = int(layer_name.split(".")[2])
        if current_layer == edit_layer:
            if isinstance(output, tuple):
                output[0][:, token_idx] += fv_vector.to(device)
                return output
            else:
                return output
        return output
    return intervention_fn


def run_with_intervention(sentence, target, edit_layer, fv_vector, 
                          model, model_config, tokenizer):
    """
    Run model on sentence with and without FV intervention.
    Returns logits from both clean and intervened runs.
    """
    device = model.device
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    
    # Clean run (no intervention)
    clean_logits = model(**inputs).logits[:, -1, :]
    
    # Intervention run
    intervention_fn = create_fv_intervention(
        edit_layer, 
        fv_vector.reshape(1, model_config['resid_dim']), 
        device
    )
    
    with TraceDict(model, layers=model_config['layer_hook_names'], 
                   edit_output=intervention_fn):
        intervened_logits = model(**inputs).logits[:, -1, :]
    
    return clean_logits, intervened_logits


def run_natural_text_intervention(sentence, edit_layer, fv_vector, 
                                   model, model_config, tokenizer, 
                                   max_new_tokens=10):
    """Generate text with and without FV intervention."""
    device = model.device
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    
    # Clean generation
    clean_output = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens, 
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Intervened generation
    intervention_fn = create_fv_intervention(edit_layer, fv_vector, device)
    
    with TraceDict(model, layers=model_config['layer_hook_names'], 
                   edit_output=intervention_fn):
        intervened_output = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return clean_output, intervened_output


print("Intervention utilities defined")

Intervention utilities defined


## Evaluation Utilities

Functions for evaluating model predictions and computing accuracy.

In [13]:
# Evaluation utilities

def decode_top_k(logits, tokenizer, k=5):
    """Decode top k tokens from logits."""
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k=k, dim=-1)
    
    results = []
    for idx, prob in zip(top_indices.squeeze(), top_probs.squeeze()):
        token = tokenizer.decode(idx)
        results.append((token, round(prob.item(), 5)))
    return results


def get_answer_token_id(query, answer, tokenizer):
    """Get the token ID for the answer in context."""
    query_ids = tokenizer(query, return_tensors='pt').input_ids[0]
    full_ids = tokenizer(query + answer, return_tensors='pt').input_ids[0]
    answer_ids = full_ids[len(query_ids):]
    return answer_ids.tolist()


def compute_token_rank(logits, target_id):
    """Compute rank of target token in logits distribution."""
    if isinstance(target_id, list):
        target_id = target_id[0]
    sorted_indices = torch.argsort(logits.squeeze(), descending=True)
    rank = (sorted_indices == target_id).nonzero().item()
    return rank


def compute_accuracy(ranks, k=1):
    """Compute top-k accuracy from rank list."""
    ranks = np.array(ranks)
    return (ranks < k).sum() / len(ranks)


print("Evaluation utilities defined")

Evaluation utilities defined


## Run Replication Experiment

Now run the full experiment:
1. Compute mean activations from ICL prompts
2. Extract function vector
3. Test in zero-shot and shuffled-label contexts

In [14]:
# Step 1: Compute mean activations for antonym task
set_seed(42)
dataset = load_task_dataset('antonym', seed=0)

print("Computing mean activations from ICL prompts...")
mean_activations = compute_mean_activations(
    dataset, model, model_config, tokenizer,
    n_icl_examples=10, n_trials=100, shuffle_labels=False
)
print(f"Mean activations shape: {mean_activations.shape}")
print(f"  Layers: {mean_activations.shape[0]}, Heads: {mean_activations.shape[1]}")
print(f"  Tokens: {mean_activations.shape[2]}, Head dim: {mean_activations.shape[3]}")

Computing mean activations from ICL prompts...


Computing mean activations:   0%|          | 0/100 [00:00<?, ?it/s]

Computing mean activations:   1%|          | 1/100 [00:00<00:39,  2.51it/s]

Computing mean activations:   3%|▎         | 3/100 [00:00<00:15,  6.15it/s]

Computing mean activations:   5%|▌         | 5/100 [00:00<00:11,  8.32it/s]

...

Computing mean activations:  95%|█████████▌| 95/100 [00:07<00:00, 12.49it/s]

Computing mean activations:  97%|█████████▋| 97/100 [00:08<00:00, 12.51it/s]

Computing mean activations:  99%|█████████▉| 99/100 [00:08<00:00, 12.53it/s]

Computing mean activations: 100%|██████████| 100/100 [00:08<00:00, 11.95it/s]

Mean activations shape: torch.Size([28, 16, 97, 256])
  Layers: 28, Heads: 16
  Tokens: 97, Head dim: 256


In [15]:
# Step 2: Compute function vector
print("Computing function vector from top 10 causal heads...")
FV, top_heads = compute_function_vector(mean_activations, model, model_config, n_top_heads=10)
print(f"Function vector shape: {FV.shape}")
print(f"Function vector dtype: {FV.dtype}")
print(f"\nTop 10 causal heads used:")
for layer, head, score in top_heads:
    print(f"  Layer {layer:2d}, Head {head:2d} (AIE: {score:.4f})")

Computing function vector from top 10 causal heads...
Function vector shape: torch.Size([1, 4096])
Function vector dtype: torch.float32

Top 10 causal heads used:
  Layer 15, Head  5 (AIE: 0.0587)
  Layer  9, Head 14 (AIE: 0.0584)
  Layer 12, Head 10 (AIE: 0.0526)
  Layer  8, Head  1 (AIE: 0.0445)
  Layer 11, Head  0 (AIE: 0.0445)
  Layer 13, Head 13 (AIE: 0.0190)
  Layer  8, Head  0 (AIE: 0.0184)
  Layer 14, Head  9 (AIE: 0.0160)
  Layer  9, Head  2 (AIE: 0.0127)
  Layer 24, Head  6 (AIE: 0.0113)


## Test Function Vector Intervention

Test the function vector in different contexts:
1. Clean ICL prompt (baseline)
2. Shuffled-label ICL prompt (labels randomized)
3. Zero-shot prompt (no demonstrations)

In [16]:
# Test with a sample from the dataset
set_seed(42)
EDIT_LAYER = 9  # Early-middle layer as recommended in the paper

# Sample ICL examples and test pair
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]
print(f"Test query: {test_pair['input']} -> {test_pair['output']}")

# Create different prompt types
# 1. Clean ICL prompt
clean_prompt_data = create_prompt_data(word_pairs, query_pair=test_pair, prepend_bos=True)
clean_prompt = build_prompt(clean_prompt_data)
print("\n1. Clean ICL Prompt:")
print(repr(clean_prompt[:200]))

Test query: static -> dynamic

1. Clean ICL Prompt:
'<|endoftext|>Q: limitless\nA: limited\n\nQ: wake\nA: sleep\n\nQ: elevate\nA: depress\n\nQ: push\nA: pull\n\nQ: stale\nA: fresh\n\nQ: static\nA:'


In [17]:
# 2. Shuffled-label ICL prompt (labels randomized)
shuffled_prompt_data = create_prompt_data(word_pairs, query_pair=test_pair, 
                                          prepend_bos=True, shuffle_labels=True)
shuffled_prompt = build_prompt(shuffled_prompt_data)
print("2. Shuffled-Label ICL Prompt:")
print(repr(shuffled_prompt[:200]))

# 3. Zero-shot prompt (no demonstrations)
zeroshot_prompt_data = create_prompt_data({'input': [], 'output': []}, 
                                          query_pair=test_pair, prepend_bos=True)
zeroshot_prompt = build_prompt(zeroshot_prompt_data)
print("\n3. Zero-Shot Prompt:")
print(repr(zeroshot_prompt))

2. Shuffled-Label ICL Prompt:
'<|endoftext|>Q: limitless\nA: sleep\n\nQ: wake\nA: fresh\n\nQ: elevate\nA: depress\n\nQ: push\nA: limited\n\nQ: stale\nA: pull\n\nQ: static\nA:'

3. Zero-Shot Prompt:
'<|endoftext|>Q: static\nA:'


In [18]:
# Test clean ICL prompt (baseline - should work without intervention)
print("=" * 60)
print("Testing Clean ICL Prompt (baseline)")
print("=" * 60)

inputs = tokenizer(clean_prompt, return_tensors='pt').to(model.device)
clean_logits = model(**inputs).logits[:, -1, :]
print(f"Query: '{test_pair['input']}' -> Expected: '{test_pair['output']}'")
print(f"Top 5 predictions: {decode_top_k(clean_logits, tokenizer, k=5)}")

Testing Clean ICL Prompt (baseline)
Query: 'static' -> Expected: 'dynamic'
Top 5 predictions: [(' dynamic', 0.82726), (' fluid', 0.01458), (' dynam', 0.0124), (' moving', 0.01145), (' static', 0.00887)]


In [19]:
# Test shuffled-label prompt with and without FV intervention
print("=" * 60)
print("Testing Shuffled-Label Prompt")
print("=" * 60)

clean_logits, intervened_logits = run_with_intervention(
    shuffled_prompt, test_pair['output'], EDIT_LAYER, FV, 
    model, model_config, tokenizer
)

print(f"Query: '{test_pair['input']}' -> Expected: '{test_pair['output']}'")
print(f"\nWithout FV (shuffled labels confuse model):")
print(f"  Top 5: {decode_top_k(clean_logits, tokenizer, k=5)}")
print(f"\nWith FV intervention:")
print(f"  Top 5: {decode_top_k(intervened_logits, tokenizer, k=5)}")

Testing Shuffled-Label Prompt
Query: 'static' -> Expected: 'dynamic'

Without FV (shuffled labels confuse model):
  Top 5: [(' dynamic', 0.04007), (' static', 0.01813), (' push', 0.01133), (' flow', 0.01099), (' motion', 0.01014)]

With FV intervention:
  Top 5: [(' dynamic', 0.37419), (' moving', 0.03447), (' fluid', 0.02847), (' mobile', 0.01873), (' motion', 0.01839)]


In [20]:
# Test zero-shot prompt with and without FV intervention
print("=" * 60)
print("Testing Zero-Shot Prompt")
print("=" * 60)

clean_logits, intervened_logits = run_with_intervention(
    zeroshot_prompt, test_pair['output'], EDIT_LAYER, FV, 
    model, model_config, tokenizer
)

print(f"Query: '{test_pair['input']}' -> Expected: '{test_pair['output']}'")
print(f"\nWithout FV (zero-shot - no task info):")
print(f"  Top 5: {decode_top_k(clean_logits, tokenizer, k=5)}")
print(f"\nWith FV intervention:")
print(f"  Top 5: {decode_top_k(intervened_logits, tokenizer, k=5)}")

Testing Zero-Shot Prompt
Query: 'static' -> Expected: 'dynamic'

Without FV (zero-shot - no task info):
  Top 5: [(' static', 0.13476), (' yes', 0.02457), (' 1', 0.02202), ('\n', 0.01759), (' no', 0.01677)]

With FV intervention:
  Top 5: [(' dynamic', 0.59716), (' static', 0.02988), (' non', 0.01073), (' Dynamic', 0.0093), (' variable', 0.00831)]


In [21]:
# Test on natural text prompt
print("=" * 60)
print("Testing Natural Text Prompt")
print("=" * 60)

natural_prompt = f'The word "{test_pair["input"]}" means'
clean_output, intervened_output = run_natural_text_intervention(
    natural_prompt, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10
)

print(f"Input: {repr(natural_prompt)}")
print(f"\nGPT-J (no FV):    {repr(tokenizer.decode(clean_output.squeeze()))}")
print(f"GPT-J + FV:       {repr(tokenizer.decode(intervened_output.squeeze()))}")

Testing Natural Text Prompt


Input: 'The word "static" means'

GPT-J (no FV):    'The word "static" means "unchanging" or "unvarying'
GPT-J + FV:       'The word "static" means "dynamic" in the sense that it is'


## Full Dataset Evaluation

Run evaluation on the test set to compute accuracy metrics for:
- Zero-shot baseline vs Zero-shot + FV
- Shuffled-label baseline vs Shuffled-label + FV

In [22]:
# Full evaluation function
def evaluate_fv_intervention(dataset, fv_vector, edit_layer, model, model_config, tokenizer,
                             n_shots=0, shuffle_labels=False, n_eval=50, seed=42):
    """
    Evaluate FV intervention on dataset.
    Returns accuracy metrics for baseline and intervened model.
    """
    set_seed(seed)
    
    clean_ranks = []
    intervened_ranks = []
    
    prepend_bos = not model_config['prepend_bos']
    n_test = min(n_eval, len(dataset['test']))
    
    for i in tqdm(range(n_test), desc=f"Evaluating (n_shots={n_shots}, shuffle={shuffle_labels})"):
        # Sample ICL examples if needed
        if n_shots > 0:
            train_idx = np.random.choice(len(dataset['train']), n_shots, replace=False)
            word_pairs = dataset['train'][train_idx]
        else:
            word_pairs = {'input': [], 'output': []}
        
        test_pair = dataset['test'][i]
        
        # Create prompt
        prompt_data = create_prompt_data(word_pairs, query_pair=test_pair,
                                         prepend_bos=prepend_bos, shuffle_labels=shuffle_labels)
        prompt = build_prompt(prompt_data)
        target = prompt_data['query_target']['output']
        
        # Get target token ID
        target_id = get_answer_token_id(prompt, target, tokenizer)
        
        # Run with intervention
        clean_logits, intervened_logits = run_with_intervention(
            prompt, target, edit_layer, fv_vector, model, model_config, tokenizer
        )
        
        # Compute ranks
        clean_rank = compute_token_rank(clean_logits, target_id)
        intervened_rank = compute_token_rank(intervened_logits, target_id)
        
        clean_ranks.append(clean_rank)
        intervened_ranks.append(intervened_rank)
    
    results = {
        'clean_top1_acc': compute_accuracy(clean_ranks, k=1),
        'clean_top3_acc': compute_accuracy(clean_ranks, k=3),
        'intervened_top1_acc': compute_accuracy(intervened_ranks, k=1),
        'intervened_top3_acc': compute_accuracy(intervened_ranks, k=3),
        'clean_ranks': clean_ranks,
        'intervened_ranks': intervened_ranks
    }
    
    return results

print("Evaluation function defined")

Evaluation function defined


In [23]:
# Run zero-shot evaluation
print("Running zero-shot evaluation...")
zeroshot_results = evaluate_fv_intervention(
    dataset, FV, EDIT_LAYER, model, model_config, tokenizer,
    n_shots=0, shuffle_labels=False, n_eval=50, seed=42
)

print("\n" + "=" * 60)
print("ZERO-SHOT RESULTS")
print("=" * 60)
print(f"Baseline Top-1 Accuracy: {zeroshot_results['clean_top1_acc']:.1%}")
print(f"Baseline Top-3 Accuracy: {zeroshot_results['clean_top3_acc']:.1%}")
print(f"FV Intervention Top-1 Accuracy: {zeroshot_results['intervened_top1_acc']:.1%}")
print(f"FV Intervention Top-3 Accuracy: {zeroshot_results['intervened_top3_acc']:.1%}")

Running zero-shot evaluation...


Evaluating (n_shots=0, shuffle=False):   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating (n_shots=0, shuffle=False):   2%|▏         | 1/50 [00:00<00:29,  1.67it/s]

Evaluating (n_shots=0, shuffle=False):   6%|▌         | 3/50 [00:00<00:09,  4.93it/s]

Evaluating (n_shots=0, shuffle=False):  10%|█         | 5/50 [00:00<00:05,  7.63it/s]

...

Evaluating (n_shots=0, shuffle=False):  90%|█████████ | 45/50 [00:03<00:00, 15.60it/s]

Evaluating (n_shots=0, shuffle=False):  94%|█████████▍| 47/50 [00:03<00:00, 15.57it/s]

Evaluating (n_shots=0, shuffle=False):  98%|█████████▊| 49/50 [00:03<00:00, 15.51it/s]

Evaluating (n_shots=0, shuffle=False): 100%|██████████| 50/50 [00:03<00:00, 13.38it/s]


ZERO-SHOT RESULTS
Baseline Top-1 Accuracy: 0.0%
Baseline Top-3 Accuracy: 8.0%
FV Intervention Top-1 Accuracy: 32.0%
FV Intervention Top-3 Accuracy: 58.0%


In [24]:
# Run shuffled-label evaluation (10-shot with shuffled labels)
print("Running shuffled-label evaluation (10-shot)...")
shuffled_results = evaluate_fv_intervention(
    dataset, FV, EDIT_LAYER, model, model_config, tokenizer,
    n_shots=10, shuffle_labels=True, n_eval=50, seed=42
)

print("\n" + "=" * 60)
print("SHUFFLED-LABEL (10-SHOT) RESULTS")
print("=" * 60)
print(f"Baseline Top-1 Accuracy: {shuffled_results['clean_top1_acc']:.1%}")
print(f"Baseline Top-3 Accuracy: {shuffled_results['clean_top3_acc']:.1%}")
print(f"FV Intervention Top-1 Accuracy: {shuffled_results['intervened_top1_acc']:.1%}")
print(f"FV Intervention Top-3 Accuracy: {shuffled_results['intervened_top3_acc']:.1%}")

Running shuffled-label evaluation (10-shot)...


Evaluating (n_shots=10, shuffle=True):   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating (n_shots=10, shuffle=True):   2%|▏         | 1/50 [00:00<00:05,  8.78it/s]

Evaluating (n_shots=10, shuffle=True):   4%|▍         | 2/50 [00:00<00:05,  8.93it/s]

Evaluating (n_shots=10, shuffle=True):   6%|▌         | 3/50 [00:00<00:05,  8.99it/s]

...

Evaluating (n_shots=10, shuffle=True):  96%|█████████▌| 48/50 [00:05<00:00,  9.01it/s]

Evaluating (n_shots=10, shuffle=True):  98%|█████████▊| 49/50 [00:05<00:00,  9.00it/s]

Evaluating (n_shots=10, shuffle=True): 100%|██████████| 50/50 [00:05<00:00,  8.99it/s]

Evaluating (n_shots=10, shuffle=True): 100%|██████████| 50/50 [00:05<00:00,  9.00it/s]


SHUFFLED-LABEL (10-SHOT) RESULTS
Baseline Top-1 Accuracy: 30.0%
Baseline Top-3 Accuracy: 50.0%
FV Intervention Top-1 Accuracy: 54.0%
FV Intervention Top-3 Accuracy: 68.0%


## Results Summary

The function vector intervention shows significant improvements:

| Setting | Baseline Top-1 | FV Top-1 | Improvement |
|---------|---------------|----------|-------------|
| Zero-shot | 0% | 32% | +32pp |
| Shuffled-label | 30% | 54% | +24pp |

These results are consistent with the paper's claims that function vectors can trigger task execution even without proper ICL demonstrations.

In [25]:
# Verify reproducibility by running the zero-shot evaluation again with the same seed
print("Verifying reproducibility with second run...")
zeroshot_results_v2 = evaluate_fv_intervention(
    dataset, FV, EDIT_LAYER, model, model_config, tokenizer,
    n_shots=0, shuffle_labels=False, n_eval=50, seed=42
)

print("\n" + "=" * 60)
print("REPRODUCIBILITY CHECK")
print("=" * 60)
print(f"Run 1 FV Top-1 Accuracy: {zeroshot_results['intervened_top1_acc']:.1%}")
print(f"Run 2 FV Top-1 Accuracy: {zeroshot_results_v2['intervened_top1_acc']:.1%}")
print(f"Results match: {zeroshot_results['intervened_top1_acc'] == zeroshot_results_v2['intervened_top1_acc']}")

Verifying reproducibility with second run...


Evaluating (n_shots=0, shuffle=False):   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating (n_shots=0, shuffle=False):   2%|▏         | 1/50 [00:00<00:05,  8.97it/s]

Evaluating (n_shots=0, shuffle=False):   6%|▌         | 3/50 [00:00<00:03, 12.88it/s]

Evaluating (n_shots=0, shuffle=False):  10%|█         | 5/50 [00:00<00:03, 13.95it/s]

...

Evaluating (n_shots=0, shuffle=False):  94%|█████████▍| 47/50 [00:03<00:00,  9.90it/s]

Evaluating (n_shots=0, shuffle=False):  98%|█████████▊| 49/50 [00:03<00:00,  9.52it/s]

Evaluating (n_shots=0, shuffle=False): 100%|██████████| 50/50 [00:04<00:00,  9.17it/s]

Evaluating (n_shots=0, shuffle=False): 100%|██████████| 50/50 [00:04<00:00, 12.43it/s]


REPRODUCIBILITY CHECK
Run 1 FV Top-1 Accuracy: 32.0%
Run 2 FV Top-1 Accuracy: 32.0%
Results match: True


In [26]:
# Save results to JSON
results_summary = {
    'task': 'antonym',
    'model': 'EleutherAI/gpt-j-6b',
    'edit_layer': EDIT_LAYER,
    'n_top_heads': 10,
    'n_icl_examples_for_fv': 10,
    'n_trials_for_fv': 100,
    'evaluation': {
        'zero_shot': {
            'n_eval': 50,
            'baseline_top1_acc': zeroshot_results['clean_top1_acc'],
            'baseline_top3_acc': zeroshot_results['clean_top3_acc'],
            'fv_top1_acc': zeroshot_results['intervened_top1_acc'],
            'fv_top3_acc': zeroshot_results['intervened_top3_acc']
        },
        'shuffled_label_10shot': {
            'n_eval': 50,
            'baseline_top1_acc': shuffled_results['clean_top1_acc'],
            'baseline_top3_acc': shuffled_results['clean_top3_acc'],
            'fv_top1_acc': shuffled_results['intervened_top1_acc'],
            'fv_top3_acc': shuffled_results['intervened_top3_acc']
        }
    },
    'reproducibility': {
        'verified': True,
        'note': 'Results are identical across two runs with same seed'
    }
}

# Save to replication directory
results_path = os.path.join(replication_dir, 'replication_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"Results saved to {results_path}")

Results saved to /net/scratch2/smallyan/function_vectors_eval/evaluation/replications/replication_results.json


## Conclusion

This replication successfully demonstrates the core claims of the Function Vectors paper:

1. **Function vectors can be extracted** from ICL prompts by computing mean activations across many trials and summing outputs of top causal attention heads.

2. **Function vectors trigger task execution** even in corrupted contexts:
   - Zero-shot: 0% → 32% accuracy with FV
   - Shuffled-labels: 30% → 54% accuracy with FV

3. **Results are reproducible** with controlled seeds - identical results across multiple runs.

The implementation faithfully recreates the methodology described in the plan without copying original code.

In [27]:
# Copy the notebook to the replication directory
import shutil

# The notebook path from the session
notebook_src = '/home/smallyan/eval_agent/notebooks/2025-12-22-21-43_CircuitAnalysis.ipynb'
notebook_dst = os.path.join(replication_dir, 'replication.ipynb')
shutil.copy(notebook_src, notebook_dst)
print(f"Notebook copied to {notebook_dst}")

Notebook copied to /net/scratch2/smallyan/function_vectors_eval/evaluation/replications/replication.ipynb
